# L4 — Repadronização anual das ativações no Escopo B do Trabalho 7

Reconstituímos os códigos SAE do Escopo B, usando os mesmos pacientes, janelas, papéis, entrada de ano real, configurações e sementes. Reproduzimos as árvores graduadas com a validação agrupada original e conferimos o parâmetro de poda registrado e reconstruímos as regras binárias com o procedimento original. Exigimos paridade com as tabelas originais antes de exportar resultados.

Para cada fator e ano posterior à referência, usamos cinco partições de pacientes. Para cada partição de teste, estimamos média e desvio padrão da ativação nas outras quatro. Transportamos a ativação para a escala do ano de referência com a transformação \(\max(0,\mu_r+\sigma_r(a-\mu_t)/\sigma_t)\). Mantemos o ano de referência intacto. Se a distribuição em um conjunto de ajuste é constante, usamos as ativações originais nessa partição e registramos a ocorrência. Um segundo método ajusta apenas a média das ativações positivas e preserva os zeros. Árvores, regras e limiares ficam congelados durante a comparação.

A transformação usa somente ativações. O protocolo pressupõe acesso a um lote de pacientes do ano corrente e estima o resultado em pacientes que ficaram fora do ajuste daquele ano. Trata-se de um diagnóstico descritivo, com efeitos pareados por fator e ano. A mediana entre fatores resume o conjunto, e a tabela também mostra quantos melhoraram. Os cálculos com linhas de pacientes permanecem na Fiji.

In [1]:
import hashlib
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import GroupKFold

import prospective_audit as audit
import prospective_graded_trees as gt

RUN_START = time.monotonic()
REFS = (2009, 2010, 2011)
THREADS = 32
SAE_DEVICE = "cuda"
FOLDS = 5
POP = Path("stats/temporal_robustness/_population/prepared.pkl")
POP_SHA256 = "c8804591c93087241b9533dce77dd54b2f1e77b23a36400527459cfa8f99add7"
ORIGINAL = Path("stats/prospective_target_audit/scope_b")
OUT = Path("stats/l4_activation_restandardization")
BUILDS = Path.home() / "rpi-runs/SAE-TCAV/pilot-ideas-followups/builds"
BRIDGE_CACHE_DIRECTORY = {
    2009: BUILDS / "all_history/unit-cache/128647232c48f725e8ff89ab7b954c3bfa46772485d948d8602709373e80c402",
    2010: BUILDS / "last_3/unit-cache/7fa1706899a8d5de2f76875e345d913e87f4a0f191f57c49886d3ae5161a076f",
    2011: BUILDS / "last_3/unit-cache/7fa1706899a8d5de2f76875e345d913e87f4a0f191f57c49886d3ae5161a076f",
}
METHODS = ("frozen", "location_scale", "positive_scale")
TREE_METRICS = ("active_share", "skill", "rmse", "calibration_intercept", "calibration_slope",
                "hurdle_auroc", "spearman_patient")
RULE_METRICS = ("active_share", "active_given_rule", "auroc_activation_rule", "rule_share")
torch.set_num_threads(THREADS)
assert torch.cuda.is_available()

files = ("graded_tree_candidates.csv", "graded_tree_year_metrics.csv",
         "binary_rule_rules.csv", "binary_rule_secondary.csv")
tree_candidates, tree_original, rule_candidates, rule_original = (
    pd.read_csv(ORIGINAL / name, float_precision="round_trip") for name in files
)
tree_candidates = tree_candidates[(tree_candidates["variant"] == "prospective") &
                                  (tree_candidates["arm"] == "main")]
tree_original = tree_original[(tree_original["variant"] == "prospective") &
                              (tree_original["arm"] == "main") &
                              (tree_original["comparison"] == "actual")]

In [2]:
def transfer_activation(reference, fit, evaluation, method):
    if method == "location_scale":
        ref_std, fit_std = reference.std(), fit.std()
        if np.ptp(reference) > 0 and np.ptp(fit) > 0:
            return np.maximum(0, reference.mean() + (evaluation - fit.mean()) * ref_std / fit_std), False
    if method == "positive_scale":
        ref_positive = reference[reference > 0]
        fit_positive = fit[fit > 0]
        if len(ref_positive) and len(fit_positive):
            return evaluation * ref_positive.mean() / fit_positive.mean(), False
    return evaluation.copy(), True


def activation_arms(reference, values, patients, in_reference_year):
    assert np.isfinite(reference).all() and np.isfinite(values).all()
    assert (reference >= 0).all() and (values >= 0).all()
    arms = {method: values.copy() for method in METHODS}
    fallbacks = {method: 0 for method in METHODS}
    if in_reference_year:
        return arms, fallbacks

    covered = np.zeros(len(values), dtype=int)
    for fit, evaluation in GroupKFold(n_splits=FOLDS).split(values, groups=patients):
        assert not np.intersect1d(patients[fit], patients[evaluation]).size
        covered[evaluation] += 1
        for method in METHODS[1:]:
            scored, fallback = transfer_activation(reference, values[fit], values[evaluation], method)
            arms[method][evaluation] = scored
            fallbacks[method] += int(fallback)
    assert (covered == 1).all()
    assert np.array_equal(arms["positive_scale"] == 0, values == 0)
    return arms, fallbacks


def record_metrics(records, keys, arms, fallbacks, get_metrics):
    for method, activation in arms.items():
        assert np.isfinite(activation).all() and (activation >= 0).all()
        records.extend({
            **keys, "method": method, "metric": metric, "estimate": value,
            "n": len(activation), "fallback_folds": fallbacks[method],
        } for metric, value in get_metrics(activation).items())


def check_baseline(result, original, id_column):
    keys = ["reference_year", "input_year", id_column, "metric"]
    frozen = result[result["method"] == "frozen"]
    expected = original[original["reference_year"].isin(REFS) &
                        original["metric"].isin(frozen["metric"].unique())]
    compared = frozen.merge(expected[keys + ["estimate"]], on=keys, how="outer",
                             validate="one_to_one", indicator=True, suffixes=("", "_original"))
    if not (compared["_merge"] == "both").all():
        raise ValueError(f"Baseline keys differ: {compared['_merge'].value_counts().to_dict()}")
    value, original_value = compared["estimate"], compared["estimate_original"]
    compared["abs_difference"] = np.abs(value - original_value)
    passed = np.isclose(value, original_value, atol=2e-5, rtol=2e-5, equal_nan=True)
    if not passed.all():
        raise ValueError(f"Baseline differs for {(~passed).sum()} of {len(passed)} values. "
                         f"Max difference: {compared['abs_difference'].max():g}")
    return {"arm": id_column, "keys": len(compared),
            "max_abs_difference": compared["abs_difference"].max(), "passed": True}


values = np.array([0., 1., 2., 3., 4., 5., 6., 7., 8., 9.])
patients = np.repeat(np.arange(5), 2)
arms, fallback = activation_arms(values, values, patients, False)
assert (arms["location_scale"] >= 0).all()
assert not any(fallback.values())
unchanged, fallback = activation_arms(values, values, patients, True)
assert all(np.array_equal(unchanged[method], values) for method in METHODS)
constant, fallback = activation_arms(values, np.zeros_like(values), patients, False)
assert fallback["location_scale"] == fallback["positive_scale"] == FOLDS
assert all(not constant[method].any() for method in METHODS)
positive_constant, fallback = activation_arms(values, np.full_like(values, 0.1), patients, False)
assert fallback["location_scale"] == FOLDS
assert np.array_equal(positive_constant["location_scale"], np.full_like(values, 0.1))

In [3]:
def reconstruct_inputs(reference, population, cohort):
    cohort_r, roles = audit.add_roles(cohort, population, reference)
    row = cohort_r["row"].to_numpy()
    year = cohort_r["year"].to_numpy()
    context = audit.context_mask(cohort_r, reference)
    evaluation = audit.evaluation_mask(cohort_r, reference)
    role = cohort_r["role"].to_numpy()
    selection = (year == reference) & np.isin(role, ["sae_discovery", "rule_discovery", "cav_fit"])
    X = population["X"]

    fitted = audit.fit_dr_tabpfn_context(
        X[row[context]], cohort_r["y_next"].to_numpy(dtype=float)[context], year[context],
        domain_origin=audit.context_first_year(reference), device="cpu", seed=audit.SEED,
        use_context_scaler=True,
    )
    assert fitted["subsample_rows"] == fitted["context_rows"]
    selected_embedding = audit.capture_scored_rows(
        fitted, X[row[selection]], year[selection], batch_size=256)["embeddings"]
    evaluation_embedding = audit.capture_scored_rows(
        fitted, X[row[evaluation]], year[evaluation], batch_size=256)["embeddings"]
    del fitted

    rows = np.concatenate([row[selection], row[evaluation]])
    embedding = np.concatenate([selected_embedding, evaluation_embedding]).astype(np.float32)
    years = np.asarray(population["year"], dtype=int)[rows]
    patients = np.asarray(population["patient_id"], dtype=str)[rows]
    row_roles = np.asarray(roles["row_role"])[rows]
    masks = gt.role_masks(row_roles, years, reference)
    masks["evaluation"] = gt.evaluation_year_mask(row_roles, years, reference)
    assert int(masks["evaluation"].sum()) == int(evaluation.sum())
    assert np.array_equal(years[masks["reference_eval"]],
                          np.full(masks["reference_eval"].sum(), reference))
    dictionaries = gt.fit_reference_saes(embedding[masks["sae_fit"]], SAE_DEVICE)
    codes = gt.encode_reference_saes(dictionaries, embedding, SAE_DEVICE)
    live = {seed: gt.live_factor_mask(a[masks["sae_fit"]]) for seed, a in codes.items()}
    candidates = gt.family_candidate_set(
        {seed: a[masks["family_fit"]] for seed, a in codes.items()},
        live, f"prospective/{reference}", SAE_DEVICE,
    )["selection"]["candidates"]
    return X[rows], years, patients, masks, candidates, gt.candidate_columns(codes, candidates)


def evaluate_reference(reference, population, cohort):
    X, years, patients, masks, candidates, raw = reconstruct_inputs(reference, population, cohort)
    ids = [candidate["family_id"] for candidate in candidates]
    known_trees = tree_candidates[tree_candidates["reference_year"] == reference].set_index("candidate")
    assert set(ids) == set(known_trees.index), f"Candidate identity mismatch in {reference}"
    fit = masks["tree_fit"]
    ref_eval = masks["reference_eval"]
    evaluation = masks["evaluation"]
    config = gt.open_bridge_cache(BRIDGE_CACHE_DIRECTORY[reference]).config
    print(f"Referência {reference}: identidades dos candidatos reproduzidas", flush=True)
    sweep = gt.run_binary_rule_arm(
        X[fit], patients[fit], list(population["feature_names"]), candidates, raw[fit],
        config["threshold_sweep"], config["rule"], reference_year=reference,
        split_seed=config["split_seed"], workers=24, rule_source=config["rule"]["rule_source"],
        recurrence=config["rule"]["recurrence"],
    )
    frozen_rules = {item["family_id"]: item for item in sweep["frozen"]}
    known_rules = rule_candidates[rule_candidates["reference_year"] == reference].set_index("family_id")
    assert set(frozen_rules) == set(known_rules.index), f"Binary rule coverage differs in {reference}"
    for family_id, entry in frozen_rules.items():
        assert entry["rule_identity"] == known_rules.loc[family_id, "rule_identity"]
        np.testing.assert_allclose(entry["cutoff"], known_rules.loc[family_id, "cutoff"], rtol=2e-5)

    records = []
    scale = gt.factor_scale_table(raw[masks["sae_fit"]])["mu_plus"].to_numpy()
    normalized = gt.normalize_activations(raw, scale)
    evaluation_years = np.unique(years[evaluation])
    for column, candidate in enumerate(ids):
        known = known_trees.loc[candidate]
        fitted_tree = gt.fit_graded_tree(X[fit], normalized[fit, column], patients[fit])
        tree = fitted_tree["tree"]
        np.testing.assert_allclose(fitted_tree["ccp_alpha"], known["ccp_alpha"], rtol=2e-5, atol=1e-9)
        ref = normalized[ref_eval, column]
        if tree.get_n_leaves() != known["leaves"]:
            raise ValueError(f"{candidate}: original leaves={known['leaves']}, replay={tree.get_n_leaves()}")
        np.testing.assert_allclose(ref.mean(), known["reference_mean"], rtol=2e-5, atol=2e-5)
        entry = frozen_rules.get(candidate)
        ref_raw = raw[ref_eval, column]
        for year in evaluation_years:
            rows = evaluation & (years == year)
            keys = {"reference_year": reference, "input_year": year, "candidate": candidate}
            at_reference = year == reference
            arms, fallback = activation_arms(ref, normalized[rows, column], patients[rows], at_reference)
            prediction = tree.predict(X[rows])

            def tree_metrics(activation):
                metrics = gt.year_factor_metrics(activation, prediction, ref.mean())
                return {name: metrics[name] for name in TREE_METRICS}

            record_metrics(records, {**keys, "arm": "graded_tree"}, arms, fallback, tree_metrics)
            if entry is None:
                continue

            arms, fallback = activation_arms(ref_raw, raw[rows, column], patients[rows], at_reference)
            X_year = X[rows]

            def rule_metrics(activation):
                return gt.frozen_rule_metrics(entry, X_year, activation)

            record_metrics(records, {**keys, "arm": "binary_rule"}, arms, fallback, rule_metrics)
    return pd.DataFrame(records)

In [4]:
population = audit.load_population(POP, POP_SHA256)
events = pd.read_feather("tidy_event_data.feather")
first_death = audit.first_death_year_by_row(population, events)
cohort = audit.landmark_cohort(population, first_death)
assert audit.cohort_check_rows(cohort)["passed"].all()
results = []
for reference in REFS:
    before = time.monotonic()
    result = evaluate_reference(reference, population, cohort)
    print(f"Referência {reference}: {len(result)} métricas agregadas em {time.monotonic() - before:.1f} s")
    results.append(result)
results = pd.concat(results, ignore_index=True)

check_rows = [
    check_baseline(results[results["arm"] == "graded_tree"], tree_original, "candidate"),
    check_baseline(results[results["arm"] == "binary_rule"].rename(columns={"candidate": "family_id"}),
                   rule_original, "family_id"),
]
pair_keys = ["arm", "reference_year", "input_year", "candidate", "metric"]
baseline = results[results["method"] == "frozen"][pair_keys + ["estimate"]]
results = results.merge(baseline.rename(columns={"estimate": "frozen_estimate"}),
                        on=pair_keys, validate="many_to_one")
results["difference"] = results["estimate"] - results["frozen_estimate"]
results["year_offset"] = results["input_year"] - results["reference_year"]
assert results.loc[results["year_offset"] == 0, "difference"].dropna().eq(0).all()

key = ["arm", "metric", "method", "year_offset"]
summary = results.groupby(key, as_index=False).agg(
    median_frozen=("frozen_estimate", "median"), median_estimate=("estimate", "median"),
    median_paired_difference=("difference", "median"), valid_pairs=("difference", "count"),
    increased=("difference", lambda x: int((x > 1e-10).sum())),
    decreased=("difference", lambda x: int((x < -1e-10).sum())),
    fallback_folds=("fallback_folds", "sum"),
)
OUT.mkdir(parents=True, exist_ok=True)
results.to_csv(OUT / "factor_year_metrics.csv", index=False)
summary.to_csv(OUT / "summary.csv", index=False)
pd.DataFrame(check_rows).to_csv(OUT / "checks.csv", index=False)
(OUT / "run_record.json").write_text(json.dumps({
    "reference_years": list(REFS), "folds": FOLDS, "predictor_device": "cpu",
    "sae_device": SAE_DEVICE, "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__,
    "wall_seconds": time.monotonic() - RUN_START, "population_sha256": POP_SHA256,
    "work7_tables_sha256": {
        name: hashlib.sha256((ORIGINAL / name).read_bytes()).hexdigest() for name in files
    },
}, indent=2) + "\n")
display(pd.DataFrame(check_rows))
display(summary[(summary["metric"].isin(["active_given_rule", "skill", "calibration_slope"])) &
                (summary["method"] != "frozen")].round(4))

/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/tabpfn/scripts/estimator/base.py:217: UserWarning: You are using a CPU device, but CUDA is available. This will likely lead to inferior performance.Fix this by setting device='cuda'.
  warnings.warn(


Initialized decoder for standard with (None, 10)  and nout 10


/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Referência 2009: identidades dos candidatos reproduzidas


/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Referência 2009: 2655 métricas agregadas em 218.3 s


... (6 outputs left out)

/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Referência 2011: 1674 métricas agregadas em 253.4 s


,arm,keys,max_abs_difference,passed
0,candidate,1491,0.0,True
1,family_id,624,0.0,True


,arm,metric,method,year_offset,median_frozen,median_estimate,median_paired_difference,valid_pairs,increased,decreased,fallback_folds
5,binary_rule,active_given_rule,location_scale,0,0.8172,0.8172,0.0000,40,0,0,0
6,binary_rule,active_given_rule,location_scale,1,0.4286,0.7082,0.1640,40,23,11,10
7,binary_rule,active_given_rule,location_scale,2,0.1129,0.6583,0.1311,40,23,12,15
8,binary_rule,active_given_rule,location_scale,3,0.0000,0.7143,0.0278,25,14,5,15
9,binary_rule,active_given_rule,location_scale,4,0.0000,0.8571,0.0000,11,5,1,12
10,binary_rule,active_given_rule,positive_scale,0,0.8172,0.8172,0.0000,40,0,0,0
11,binary_rule,active_given_rule,positive_scale,1,0.4286,0.6667,0.0196,40,20,6,10
12,binary_rule,active_given_rule,positive_scale,2,0.1129,0.6479,0.0196,40,20,4,15
13,binary_rule,active_given_rule,positive_scale,3,0.0000,0.4857,0.0000,25,10,2,15
14,binary_rule,active_given_rule,positive_scale,4,0.0000,0.0000,0.0000,11,3,0,12
